# LLM Zoomcamp 2026 - dlt Workshop Homework



## 1. Imports and Configuration


In [2]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Any

import duckdb
import requests
from dotenv import load_dotenv
from minsearch import Index
from pydantic_ai import Agent, RunContext

load_dotenv()

HOMEWORK_QUESTION = 'How do I run Ollama locally?'
DUCKDB_PATH = r'C:\Users\Valentina Cruz DP\Documents\Camila_bootcamp\llm-zoomcamp-2026-code\zoompcamp-h5\logfire.duckdb'
DATASET_NAME = 'agent_traces'

def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'Missing required environment variable: {name}')
    return value

{name: bool(os.getenv(name)) for name in ['OPENAI_API_KEY', 'LOGFIRE_TOKEN', 'LOGFIRE_READ_TOKEN']}


{'OPENAI_API_KEY': True, 'LOGFIRE_TOKEN': True, 'LOGFIRE_READ_TOKEN': True}

## 2. Agent Setup


In [3]:
INSTRUCTIONS = '''
You're a course teaching assistant. You're given a question from a course student
and your task is to answer it. If you want to look up information, use the search
function. Use as many keywords from the user question as possible when making
first requests. Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic
question. If you can't answer the question using FAQ, don't do it yourself. Only
use the facts from the FAQ database. At the end, ask if there are other areas
that the user wants to explore.
'''.strip()

@dataclass
class SearchDeps:
    index: Index

faq_agent = Agent(
    'openai:gpt-5.4-mini',
    deps_type=SearchDeps,
    instructions=INSTRUCTIONS,
)

@faq_agent.tool
def search(ctx: RunContext[SearchDeps], query: str) -> str:
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}
    return ctx.deps.index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )


## 3. Load FAQ Data


In [4]:
def load_faq_data() -> list[dict[str, Any]]:
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url, timeout=30)
    response.raise_for_status()

    documents: list[dict[str, Any]] = []
    url_prefix = 'https://datatalks.club/faq'
    for course in response.json():
        course_response = requests.get(f'{url_prefix}{course["path"]}', timeout=30)
        course_response.raise_for_status()
        documents.extend(course_response.json())

    return documents

def build_index(documents: list[dict[str, Any]]) -> Index:
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course'],
    )
    index.fit(documents)
    return index

documents = load_faq_data()
deps = SearchDeps(index=build_index(documents))
len(documents)


1380

## 4. Run Agent and Create Logfire Trace


In [5]:
require_env('OPENAI_API_KEY')
require_env('LOGFIRE_TOKEN')

import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

with logfire.span('homework-run-q1'):
    result = await faq_agent.run(HOMEWORK_QUESTION, deps=deps)
print(result.output)

print('Question 1: open this run in Logfire and count spans.')


18:06:38.953 homework-run-q1
18:06:38.969   faq_agent run
18:06:38.972     chat gpt-5.4-mini


Logfire project URL: https://logfire-us.pydantic.dev/camila-cruz-depaula/llm-zoomcamp-dt

18:06:42.696     running tool: search
18:06:42.709     chat gpt-5.4-mini
18:06:43.746     running tool: search
18:06:43.757     chat gpt-5.4-mini
To run Ollama locally for the course:

1. Install Ollama from: https://ollama.com/download  
   - macOS: install the `.pkg`
   - Windows: install the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model locally:
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. Check that the local server is running:
   ```bash
   curl http://localhost:11434
   ```
   You should see a response like:
   ```json
   {"models": [...]}
   ```

4. If you need to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```

If you get a connection refuse

## 5. Load Logfire Traces into DuckDB with dlt


In [6]:
import requests
import dlt
from dotenv import load_dotenv
load_dotenv()

LOGFIRE_BASE = 'https://logfire-us.pydantic.dev'
LOGFIRE_READ_TOKEN = os.getenv('LOGFIRE_READ_TOKEN')

def logfire_query_source(sql: str):
    @dlt.resource(name='query', write_disposition='replace')
    def query_resource():
        response = requests.post(
            f'{LOGFIRE_BASE}/v2/query',
            json={
                'sql': sql,
                'min_timestamp': '2020-01-01T00:00:00Z',
            },
            headers={
                'Authorization': f'Bearer {LOGFIRE_READ_TOKEN}',
                'Content-Type': 'application/json',
                'Accept': 'application/json',
            },
        )
        response.raise_for_status()
        data = response.json()['data']
        print(f'DEBUG: API devolvió {len(data)} rows')
        if data:
            print(f'DEBUG: columns: {list(data[0].keys())}')
        yield from data
    return query_resource

sql = '''
SELECT * FROM records
ORDER BY start_timestamp DESC
LIMIT 1000
'''.strip()

pipeline = dlt.pipeline(
    pipeline_name='logfire_pipeline',
    destination=dlt.destinations.duckdb(DUCKDB_PATH),
    dataset_name=DATASET_NAME,
)
load_info = pipeline.run(logfire_query_source(sql)())
print(load_info)


DEBUG: API devolvió 56 rows
DEBUG: columns: ['created_at', 'start_timestamp', 'end_timestamp', 'duration', 'trace_id', 'span_id', 'kind', 'level', 'parent_span_id', 'span_name', 'message', 'log_body', 'otel_status_code', 'otel_status_message', 'otel_links', 'otel_events', 'is_exception', 'tags', 'exception_message', 'exception_type', 'exception_stacktrace', 'attributes_json_schema', 'attributes', 'otel_scope_name', 'otel_scope_version', 'otel_scope_attributes', 'service_namespace', 'service_name', 'service_version', 'service_instance_id', 'process_pid', 'otel_resource_attributes', 'telemetry_sdk_name', 'telemetry_sdk_language', 'telemetry_sdk_version', 'deployment_environment', 'http_response_status_code', 'url_path', 'url_query', 'url_full', 'http_route', 'http_method', 'project_id', 'day']


2026-07-27 18:06:49,393|[WARNING]|19076|20364|dlt|validate.py|verify_normalized_table:113|In schema `logfire`: The following columns in table 'query' did not receive any data during this load and therefore could not have their types inferred:
  - attributes__model_request_parameters__output_object
  - attributes__model_request_parameters__prompted_output_template
  - attributes__model_request_parameters__thinking
  - attributes_json_schema
  - deployment_environment
  - http_method
  - http_response_status_code
  - http_route
  - log_body
  - url_full
  - url_path
  - url_query

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'attributes__model_request_parameters__output_object': {'data_type': 'text'}})

2026-07-27 18:06:49,394|[WARNING]|19076|20364|dlt|validate.py|verify_normalized_table:113|In schema `logfire

Pipeline logfire_pipeline load step completed in 1.54 seconds
1 load package(s) were loaded to destination duckdb and into dataset agent_traces
The duckdb destination used duckdb:///C:\Users\Valentina Cruz DP\Documents\Camila_bootcamp\llm-zoomcamp-2026-code\zoompcamp-h5\logfire.duckdb location to store data
Load package 1785193608.1345503 is LOADED and contains no failed jobs


## 6. Question 2 - Count Tables


In [7]:
con = duckdb.connect(DUCKDB_PATH)

table_count = con.sql(
    f'''
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_schema = '{DATASET_NAME}'
    '''
).fetchone()[0]

tables = con.sql(
    f'''
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = '{DATASET_NAME}'
    ORDER BY table_name
    '''
).fetchall()

print(f'Question 2 table count: {table_count}')
for (table_name,) in tables:
    print(table_name)


Question 2 table count: 25
_dlt_loads
_dlt_pipeline_state
_dlt_version
query
query__attributes__gen_ai_input_messages
query__attributes__gen_ai_input_messages__parts
query__attributes__gen_ai_input_messages__parts__result
query__attributes__gen_ai_output_messages
query__attributes__gen_ai_output_messages__parts
query__attributes__gen_ai_response_finish_reasons
query__attributes__gen_ai_system_instructions
query__attributes__gen_ai_tool_call_result
query__attributes__gen_ai_tool_definitions
query__attributes__gen_ai_tool_definitions__parameters__required
query__attributes__logfire_metrics__gen_ai_client_token_usage__details
query__attributes__logfire_metrics__operation_cost__details
query__attributes__logfire_scrubbed
query__attributes__logfire_scrubbed__path
query__attributes__model_request_parameters__function_tools
query__attributes__model_request_parameters__function_tools__parameters_json_schema__required
query__attributes__model_request_parameters__instruction_parts
query__attribu

## 7. Question 3 - Sum Input Tokens


In [8]:
candidates = con.sql(
    f'''
    SELECT table_schema, table_name, column_name
    FROM information_schema.columns
    WHERE table_schema = '{DATASET_NAME}'
      AND lower(column_name) LIKE '%input%token%'
    ORDER BY table_name, column_name
    '''
).fetchall()

candidates


[('agent_traces',
  'query',
  'attributes__gen_ai_aggregated_usage_cache_read_input_tokens'),
 ('agent_traces', 'query', 'attributes__gen_ai_aggregated_usage_input_tokens'),
 ('agent_traces', 'query', 'attributes__gen_ai_usage_cache_read_input_tokens'),
 ('agent_traces', 'query', 'attributes__gen_ai_usage_input_tokens')]

In [9]:
selects = []
for schema, table, column in candidates:
    selects.append(
        f'SELECT SUM(TRY_CAST("{column}" AS BIGINT)) AS tokens '
        f'FROM "{schema}"."{table}"'
    )

if not selects:
    print('No input-token column found in normalized tables.')
else:
    token_sql = 'SELECT SUM(tokens) FROM (' + ' UNION ALL '.join(selects) + ')'
    token_total = con.sql(token_sql).fetchone()[0]
    print(f'Question 3 input tokens total: {token_total}')


Question 3 input tokens total: 53632
